In [29]:
import pandas as pd
import numpy as np

In [85]:
tweet_data_full = pd.read_csv('firstsquawk_raw_data_full.csv', dtype={0: str})
tweet_data_full['New York Date'] = pd.to_datetime(
    tweet_data_full['New York Date'], 
    format='%Y-%m-%d'
)

# set UTC = True to standardise timestamps in case offset is different between tweets
tweet_data_full['New York Timestamp'] = pd.to_datetime(
    tweet_data_full['New York Timestamp'], 
    format='%Y-%m-%d %H:%M:%S%z', 
    utc=True
)
# convert to NY time
tweet_data_full['New York Timestamp'] = tweet_data_full['New York Timestamp'].dt.tz_convert('America/New_York')

In [87]:
ny_time = tweet_data_full['New York Timestamp'].dt.tz_convert('America/New_York')
is_weekday = ny_time.dt.weekday.between(0, 4)

# naive assumption that market hours are just between 0930 to 1600.
# in reality there are market days that are just half days
is_market_hours = ny_time.dt.time.between(pd.to_datetime('09:30:00').time(), 
                                          pd.to_datetime('16:00:00').time())

market_hours_mask = is_weekday & is_market_hours

df_market_hours = tweet_data_full[market_hours_mask].copy()
df_outside_hours = tweet_data_full[~market_hours_mask].copy()

print(f"Tweets during market hours: {len(df_market_hours)}")
print(f"Tweets outside market hours: {len(df_outside_hours)}")

Tweets during market hours: 42936
Tweets outside market hours: 124388


In [107]:
df_outside_hours.tail()

,Tweet ID,New York Date,New York Timestamp,Headline Text,Twitter URL
167319,2062855285702791561,2026-06-05,2026-06-05 07:14:21-04:00,CHINA'S PREMIER LI QIANG CONDUCTS A STATE COUN...,https://twitter.com/FirstSquawk/status/2062855...
167320,2062857518074966245,2026-06-05,2026-06-05 07:23:04-04:00,U\.S\. INDO\-PACIFIC COMMAND: INTERNATIONAL WA...,https://twitter.com/FirstSquawk/status/2062857...
167321,2062857498487607373,2026-06-05,2026-06-05 07:23:09-04:00,U\.S\. INDO\-PACIFIC COMMAND: WE WILL CONTINUE...,https://twitter.com/FirstSquawk/status/2062857...
167322,2062857472294220229,2026-06-05,2026-06-05 07:23:14-04:00,"U\.S\. INDO\-PACIFIC COMMAND: OVERNIGHT, U\.S\...",https://twitter.com/FirstSquawk/status/2062857...
167323,2062862167297479030,2026-06-05,2026-06-05 07:42:26-04:00,IRANIAN NAVY CLAIMS WARNING SHOTS WERE FIRED A...,https://twitter.com/FirstSquawk/status/2062862...


In [99]:
df_market_hours.tail()

,Tweet ID,New York Date,New York Timestamp,Headline Text,Twitter URL
167120,2062618408471200041,2026-06-04,2026-06-04 15:32:56-04:00,TRUMP ANNOUNCES $700M COAL INVESTMENT\.,https://twitter.com/FirstSquawk/status/2062618...
167121,2062618848097190366,2026-06-04,2026-06-04 15:34:11-04:00,PENTAGON WEIGHS CANCELLING MISSILE SALE TO GER...,https://twitter.com/FirstSquawk/status/2062618...
167122,2062618989763981733,2026-06-04,2026-06-04 15:35:22-04:00,TRUMP SAYS HE HAD A PRODUCTIVE MEETING WITH GM...,https://twitter.com/FirstSquawk/status/2062618...
167123,2062622380837802305,2026-06-04,2026-06-04 15:48:11-04:00,TRUMP ANNOUNCES PROMENADE EXTENSION AT LINCOLN...,https://twitter.com/FirstSquawk/status/2062622...
167124,2062625147455258872,2026-06-04,2026-06-04 15:59:49-04:00,TRUMP SAYS A ZELENSKYY\-PUTIN MEETING WOULD BE...,https://twitter.com/FirstSquawk/status/2062625...


In [109]:
market_ny_time = df_market_hours['New York Timestamp'].dt.tz_convert('America/New_York')
outside_ny_time = df_outside_hours['New York Timestamp'].dt.tz_convert('America/New_York')

# 2. Define our market hour boundaries
market_start = pd.to_datetime('09:30:00').time()
market_end = pd.to_datetime('16:00:00').time()

assert (market_ny_time.dt.weekday.max() <= 4), "Error: Found weekend data in market hours!"

# Check 2: Ensure all times are >= 9:30 AM and <= 4:00 PM
assert (market_ny_time.dt.time.min() >= market_start), f"Error: Found time earlier than 9:30 AM: {market_ny_time.dt.time.min()}"
assert (market_ny_time.dt.time.max() <= market_end), f"Error: Found time later than 4:00 PM: {market_ny_time.dt.time.max()}"

In [111]:
# Save the full, cleaned dataset
tweet_data_full.to_csv('firstsquawk_cleaned_full.csv', index=False)

# Save the market hours dataset
df_market_hours.to_csv('firstsquawk_market_hours.csv', index=False)

# Save the outside market hours dataset
df_outside_hours.to_csv('firstsquawk_outside_hours.csv', index=False)

print("All 3 dataframes have been successfully saved as CSVs!")

All 3 dataframes have been successfully saved as CSVs!
